In [101]:
import asyncio
import re
import pyperclip
from playwright.async_api import async_playwright

SEARCH_URL = "http://localhost:3000"


# =========================
# Normalize URL helper
# =========================
def normalize_url(url: str) -> str:
    url = re.sub(r"\s*›\s*", "/", url)
    url = re.sub(r"/{2,}", "/", url)
    url = url.replace("https:/", "https://")
    return url.strip()


# =========================
# Parse DuckDuckGo clipboard
# =========================
def parse_duckduckgo(raw_text: str):
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    results = []

    i = 0
    while i < len(lines):
        line = lines[i]

        if re.search(r"https?://", line):
            url = normalize_url(line)
            title = None
            content = None

            j = i + 1
            while j < len(lines):
                if not re.search(r"https?://", lines[j]):
                    if title is None:
                        title = lines[j]
                    elif content is None:
                        content = lines[j]
                        break
                j += 1

            if title:
                results.append({
                    "title": title,
                    "url": url,
                    "content": content
                })

            i = j
        else:
            i += 1

    return results


# =========================
# Parse Google clipboard
# =========================
def parse_google(raw_text: str):
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    results = []

    for idx in range(len(lines)):

        line = lines[idx]

        # Google URL line often contains breadcrumb "›"
        if "https://" in line or "›" in line:

            url = normalize_url(line)

            # Title is usually 2 lines above
            title = lines[idx - 2] if idx >= 2 else None

            # Snippet usually after "PDF"
            snippet = None
            if idx + 1 < len(lines) and lines[idx + 1] == "PDF":
                if idx + 2 < len(lines):
                    snippet = lines[idx + 2]
            else:
                if idx + 1 < len(lines):
                    snippet = lines[idx + 1]

            # Only keep pdf-related links
            if url.startswith("https://") and "pdf" in url.lower():
                results.append({
                    "title": title,
                    "url": url,
                    "content": snippet
                })

    return results


# =========================
# Main search engine
# =========================
async def searchEngine(query: str, headless=True, google_engine=False):

    # If Google mode → use DuckDuckGo bang
    if google_engine:
        query = "!g " + query

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context()
        page = await context.new_page()

        # ===== Open search engine =====
        await page.goto(SEARCH_URL)
        await asyncio.sleep(5)

        # ===== Search =====
        print(f"🔍 Searching: {query}")
        await page.keyboard.press("Control+L")
        await page.keyboard.press("Control+A")
        await page.keyboard.press("Backspace")
        await page.keyboard.type(query, delay=20)
        await page.keyboard.press("Enter")

        print("⏳ Waiting for results...")
        await asyncio.sleep(10)
        if google_engine:
            await page.keyboard.press("Control+L")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Control+C")
            await asyncio.sleep(1)
            raw_link = pyperclip.paste()
            print("raw_link:", raw_link)

            await asyncio.sleep(3)
            await page.keyboard.press("Backspace")

            view_source_query = "view-source:" + raw_link
            await page.keyboard.type(view_source_query, delay=20)
            await page.keyboard.press("Enter")

            print("⏳ Waiting access view-source...")
            await asyncio.sleep(10)
            print("📋 Copying results...")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Control+C")
            await asyncio.sleep(1)
            raw_text = pyperclip.paste()

            # Save to file
            with open("google_result.html", "w", encoding="utf-8") as f:
                f.write(raw_text)

            print("✅ Saved clipboard content to google_result.html")

            

        else: 

            # ===== Copy results =====
            print("📋 Copying results...")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Control+C")
            await asyncio.sleep(1)

            raw_text = pyperclip.paste()

        await browser.close()

    print(f"✅ Clipboard captured ({len(raw_text)} chars)")

    # ===== Parse depending on mode =====
    if google_engine:
        print("🌍 Parsing in GOOGLE mode...")
        return parse_google(raw_text)
    else:
        print("🦆 Parsing in DUCKDUCKGO mode...")
        return parse_duckduckgo(raw_text)


# =========================
# Wrapper main
# =========================
async def main(query, headless=True, google_engine=False):
    results = await searchEngine(query, headless, google_engine)
    return results

In [83]:
import os
from langchain_openai import ChatOpenAI

# Configure ProxyPal for ChatGPT
os.environ["OPENAI_API_KEY"] = "proxypal-local"
os.environ["OPENAI_API_BASE"] = "http://localhost:8317/v1"

import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class SearchReportValidator:
    def __init__(self, model_name="gemini-2.5-flash"):
        self.model_name = model_name
        self.llm = ChatOpenAI(model=model_name, temperature=0.0)

    def run_chatgpt(self, user_prompt: str) -> str:
        prompt = ChatPromptTemplate.from_messages([
            ("user", "{input}")
        ])
        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"input": user_prompt})

    def best_report(self, query: str, results: list) -> dict:
        """
        Return ONLY the single best matching + newest report
        from search results.
        """

        user_prompt = f"""
        You are an AI agent that selects the BEST report result.

        Query:
        {query}

        Search results:
        {json.dumps(results, indent=2)}

        Task:
        - Select ONLY ONE result that best matches the query intent
        - Prefer official PDF reports
        - Prefer the newest report (latest year/date in title/content/url)
        - If no true match exists, still return the closest available report

        Output format (STRICT JSON ONLY):

        {{
          "url": "...",
          "title": "...",
          "category": "ir_report / governance_report / other",
          "detected_date": "YYYY-MM-DD or YYYY or null",
          "why_best": "short explanation"
        }}

        Do NOT output anything outside JSON.
        """

        raw = self.run_chatgpt(user_prompt)

        # clean markdown fences
        raw = raw.strip()
        if raw.startswith("```json"):
            raw = raw[7:]
        if raw.startswith("```"):
            raw = raw[3:]
        if raw.endswith("```"):
            raw = raw[:-3]

        raw = raw.strip()

        return json.loads(raw)

validator = SearchReportValidator()

In [84]:
file_paths = {
    "hd.eneos.co.jp": {
        "ir_report": [
            "data/hd.eneos.co.jp/00.pdf",
        ],
        "governance_report": [
            "data/hd.eneos.co.jp/system_governance_report.pdf"
        ],
    },

    "mitsubishicorp.com": {
        "ir_report": [
            "data/mitsubishicorp.com/all.pdf",
        ],
        "governance_report": [
            "data/mitsubishicorp.com/governance_report_j.pdf"
        ],
    },

    "lasertec.co.jp": {
        "ir_report": [
            "data/lasertec.co.jp/6920_ir_material_for_fiscal_ym15_192733_00.pdf",
        ],
        "governance_report": [
            "data/lasertec.co.jp/6920_tdnet_2691142_00.pdf"
        ],
    },

    "itochu.co.jp": {
        "ir_report": [
            "data/itochu.co.jp/ja_ir_download___icsFiles_afieldfile_2025_09_05_ar2025J.pdf",
        ],
        "governance_report": [
            "data/itochu.co.jp/ja_files_corporate_governance.pdf"
        ],
    },

    "casio.com": {
        "ir_report": [
            "data/casio.com/content_dam_casio_global_corporate_ir_library_annual_2025_integrated-2025.pdf",
        ],
        "governance_report": [
            "data/casio.com/disclosure_20251224_20251218522473.pdf"
        ],
    },

    "boi.jp": {
        "ir_report": [
            "data/boi.jp/xcontents_AS80485_6a8dfa8d_7071_47c1_ac15_9a4be8a876b8_140120251113500915.pdf",
            "data/boi.jp/xcontents_AS80485_2e4d59dd_a617_4b86_b05d_2fbfda9ec6a2_S100XD6Y.pdf",
        ],
        "governance_report": [
            "data/boi.jp/files_tdnet_140120251119506073.pdf"
        ],
    },

    "mol.co.jp": {
        "ir_report": [
            "data/mol.co.jp/ja_ir_library_integrated_report_main_01_teaserItems2_0_linkList_0_link__J_MOL_20REPORT_2025.pdf"
        ],
        "governance_report": [
            "data/mol.co.jp/sustainability_governance_corporate_policy_pdf_governance-report.pdf"
        ],
    },

    "nintendo.co.jp": {
        "ir_report": [
            "data/nintendo.co.jp/ir_pdf_2025_annual2503e.pdf"
        ],
        "governance_report": [
            "data/nintendo.co.jp/ir_en_management_governance.pdf"
        ],
    },

    "global.toyota": {
        "ir_report": [
            "data/global.toyota/pages_global_toyota_ir_library_annual_2024_001_integrated_en.pdf"
        ],
        "governance_report": [
            "data/global.toyota/files_tdnet_140120250721517384.pdf"
        ],
    },

    "dena.com": {
        "ir_report": [
            "data/dena.com/00_2025_en.pdf"
        ],
        "governance_report": [
            "data/dena.com/files_tdnet_140120251112598427.pdf"
        ],
    },

    "lycorp.co.jp": {
        "ir_report": [
            "data/lycorp.co.jp/integrated_report_FY2024_jp.pdf"
        ],
        "governance_report": [
            "data/lycorp.co.jp/files_tdnet_140120251226527170.pdf"
        ],
    },

    "shinetsu.co.jp": {
        "ir_report": [
            "data/shinetsu.co.jp/統合報告書2025.pdf"
        ],
        "governance_report": [
            "data/shinetsu.co.jp/files_tdnet_140120251223524981.pdf"
        ],
    },

    "bridgestone.co.jp": {
        "ir_report": [
            "data/bridgestone.co.jp/ir2025_single.pdf"
        ],
        "governance_report": [
            "data/bridgestone.co.jp/files_tdnet_140120251031583941.pdf"
        ],
    },

    "capcom.co.jp": {
        "ir_report": [
            "data/capcom.co.jp/ir_english_data_pdf_annual_2025_annual_2025_01.pdf"
        ],
        "governance_report": [
            "data/capcom.co.jp/files_tdnet_140120260106529563.pdf"
        ],
    },

    "fastretailing.com": {
        "ir_report": [
            "data/fastretailing.com/jp_ir_library_pdf_ar2024.pdf"
        ],
        "governance_report": [
            "data/fastretailing.com/jp_about_governance_pdf_governance_report.pdf"
        ],
    },

    "group.softbank": {
        "ir_report": [
            "data/group.softbank/media_Project_sbg_sbg_pdf_ir_financials_annual_reports_annual-report_fy2025_ja.pdf"
        ],
        "governance_report": [
            "data/group.softbank/media_Project_sbg_sbg_pdf_about_corporate_governance_governance_20250704_01_ja.pdf"
        ],
    },

    "daiichisankyo.co.jp": {
        "ir_report": [
            "data/daiichisankyo.co.jp/files_investors_library_annual_report_index_VR2025_ds_vr2025_all_1119.pdf"
        ],
        "governance_report": [
            "data/daiichisankyo.co.jp/files_tdnet_140120251218521839.pdf"
        ],
    },

    "kajima.co.jp": {
        "ir_report": [
            "data/kajima.co.jp/english_sustainability_report_2025_pdf_ir_e_all_2.pdf"
        ],
        "governance_report": [
            "data/kajima.co.jp/files_tdnet_140120250612588145.pdf"
        ],
    },

    "mhi.com": {
        "ir_report": [
            "data/mhi.com/jp_finance_library_annual_pdf_report_2025.pdf"
        ],
        "governance_report": [
            "data/mhi.com/files_tdnet_140120250624597698.pdf"
        ],
    },

    "mitsuifudosan.co.jp": {
        "ir_report": [
            "data/mitsuifudosan.co.jp/corporate_ir_library_integratedreport_pdf_IR2025_ja.pdf"
        ],
        "governance_report": [
            "data/mitsuifudosan.co.jp/files_tdnet_140120250514552438.pdf"
        ],
    },

    "koeitecmo.co.jp": {
        "ir_report": [
            "data/koeitecmo.co.jp/files_tdnet_140120251104586205.pdf"
        ],
        "governance_report": [
            "data/koeitecmo.co.jp/files_tdnet_140120250529573336.pdf"
        ],
    },

    "toei-anim.co.jp": {
        "ir_report": [
            "data/toei-anim.co.jp/en_ir_library_Report_main_00_teaserItems1_0_linkList_0_link_PEROS_20REPORT_202024_en_open.pdf"
        ],
        "governance_report": [
            "data/toei-anim.co.jp/files_tdnet_140120250604582039.pdf"
        ],
    },

    "mufg.jp": {
        "ir_report": [
            "data/mufg.jp/dam_ir_presentation_2025_pdf_slides2509_ja.pdf"
        ],
        "governance_report": [
            "data/mufg.jp/files_tdnet_140120251107591892.pdf"
        ],
    },

    "jfe-holdings.co.jp": {
        "ir_report": [
            "data/jfe-holdings.co.jp/common_pdf_investor_library_group-report_2025_all_A4.pdf"
        ],
        "governance_report": [
            "data/jfe-holdings.co.jp/en_common_pdf_company_info_corporate-governance.pdf"
        ],
    },

    "advantest.com": {
        "ir_report": [
            "data/advantest.com/E_all_IAR2025.pdf"
        ],
        "governance_report": [
            "data/advantest.com/files_tdnet_140120251126509485.pdf"
        ],
    },
}

In [102]:
site = "kajima.co.jp"
query = f"site:{site} filetype:pdf IR report"
headless = False
google_engine = True
results_search = await main(query, headless=headless, google_engine = google_engine)
output = validator.best_report(query, results_search)
print(output)

🔍 Searching: !g site:kajima.co.jp filetype:pdf IR report
⏳ Waiting for results...
⏳ Waiting for results...
raw_link: https://www.google.com/search?hl=en&q=site%3Akajima.co.jp%20filetype%3Apdf%20ir%20report
raw_link: https://www.google.com/search?hl=en&q=site%3Akajima.co.jp%20filetype%3Apdf%20ir%20report
⏳ Waiting access view-source...
⏳ Waiting access view-source...
📋 Copying results...
📋 Copying results...
✅ Saved clipboard content to google_result.html
✅ Clipboard captured (372366 chars)
🌍 Parsing in GOOGLE mode...
✅ Saved clipboard content to google_result.html
✅ Clipboard captured (372366 chars)
🌍 Parsing in GOOGLE mode...
{'url': None, 'title': None, 'category': 'ir_report', 'detected_date': None, 'why_best': 'No search results were provided to select from, therefore no report could be chosen.'}
{'url': None, 'title': None, 'category': 'ir_report', 'detected_date': None, 'why_best': 'No search results were provided to select from, therefore no report could be chosen.'}


In [86]:
results_search

[{'title': 'KAJIMA Integrated Report 2025（Single-page）',
  'url': 'https://www.kajima.co.jp/pdf/ir_e_all_2',
  'content': 'After receiving reports from the. IR department on our market valuation, including details of dialogue with investors, the Board reviews financial strategies.'},
 {'title': 'Integrated Report - 2023',
  'url': 'https://www.kajima.co.jp/report/pdf/ir_e_all',
  'content': "The report is prepared to help readers understand the Group's initiatives for increasing corporate value and for creating value with the aim of building a more."},
 {'title': 'IR Activities Social Contribution Activities',
  'url': 'https://www.kajima.co.jp/pdf/ir_e_p107-108',
  'content': "Based on the Kajima's Corporate Philosophy of striving to continually advance our business operations and contribute to society,."},
 {'title': 'FY2025 First Half Financial Results',
  'url': 'https://www.kajima.co.jp/ir/presentation/pdf',
  'content': 'Nov 11, 2025 — Consolidated results showed higher revenues 